# Step 6 — Station x Month Day/Night Analysis (time x space)

**Input**: `output/processed_hourly_bike_counts_time_categories.csv.gz` (Step 4 output,
per-station per-hour, untouched by any step in between), `output/station_metadata.csv`

**Output**:
- `output/station_month_day_night.csv` (328 stations x 14 months, weighted day/night metrics)
- 4 PNGs under `output/charts_v2/`:
  1. `daily_day_night_line.png` -- network-wide daily day vs night line chart (424 days)
  2. `station_month_ratio_heatmap.png` -- station x month day_night_ratio heatmap
  3. `station_month_day_volume_heatmap.png` -- station x month daytime volume heatmap
  4. `station_month_night_volume_heatmap.png` -- station x month nighttime volume heatmap

Note: an interactive, more detailed version of this same day/night analysis (heatmap + linked
map + seasonal-index maps) was later built in the Combined Dashboard notebook. These 4 static
PNGs are kept here for completeness but the interactive dashboard is the primary deliverable for
this topic going forward.

## What this version does

1. **Station x month is the core output**: the main product is a day/night metric per station
   per month, keeping both the time (month) and space (station) dimensions -- this is the main
   point of this step, as opposed to collapsing all 328 stations into one network-wide number.
2. **A daily line chart, not a monthly one**: the network-wide day/night comparison uses 424 days
   of daily data rather than 14 heavily-smoothed monthly points, so real time-series dynamics
   (e.g. a sudden dip on a specific day) are visible.
3. **Extreme-value handling** (the same class of problem seen in Step 3 shows up again here):
   computing `avg_day_count_per_hour / avg_night_count_per_hour` directly, some low-traffic
   stations in some months end up dividing by something close to zero, producing an unreasonable
   ratio as high as 388 -- this isn't a data error, it's that the ratio itself becomes
   statistically unstable when the sample size is tiny (e.g. only 0-2 bikes counted at night that
   whole month). Here, `total_day_count < 10` or `total_night_count < 10` (fewer than 10 total
   counts that month for that station) is used as a threshold; ratios that don't clear it are
   marked missing, showing up gray on the heatmap instead of being mistaken for a genuine hotspot.
4. **Ratio heatmap uses a log scale + diverging colorscale (centered at ratio=1)**: a ratio is a
   multiplicative quantity (3x vs 1/3x should feel equally extreme in opposite directions), and a
   linear colorscale would let extreme values blow out the range and hide the mid-range
   differences. Mapping log10(ratio) onto a diverging colorscale preserves the "more day vs more
   night" direction while not letting a few real-but-extreme stations (e.g. one station with 3,832
   daytime vs only 29 nighttime counts -- enough sample size to be a real signal, not noise) wash
   out the overall contrast.
5. **Day/Night volume heatmaps use a log scale (sequential, one direction)**: raw traffic volume
   is right-skewed (a handful of major-corridor stations far outweigh the rest), so a linear
   colorscale would only show a few bright spots with everything else looking flat and dark. A log
   scale reveals the differences among "ordinary" stations too.


In [1]:
import pandas as pd
import numpy as np
import os
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors

plt.rcParams['font.family'] = 'DejaVu Sans'  # English-only charts, no CJK font needed
plt.rcParams['figure.dpi'] = 100
plt.rcParams['savefig.bbox'] = 'tight'

PROCESSED_PATH = "output/processed_hourly_bike_counts_time_categories.csv.gz"
STATION_META_PATH = "output/station_metadata.csv"
STATION_MONTH_OUT = "output/station_month_day_night.csv"
CHART_DIR = "output/charts_v2"
os.makedirs(CHART_DIR, exist_ok=True)

# Sample-size threshold: fewer than this many total day or night counts that month is considered
# statistically unreliable
LOW_SAMPLE_THRESHOLD = 10

In [2]:
df = pd.read_csv(
    PROCESSED_PATH,
    dtype={'station_id': 'int64', 'date': 'str', 'month': 'str',
           'bike_count_hourly': 'float64', 'daylight_fraction': 'float64',
           'day_count_weighted': 'float64', 'night_count_weighted': 'float64'},
    usecols=['station_id', 'date', 'month', 'bike_count_hourly', 'daylight_fraction',
             'day_count_weighted', 'night_count_weighted'],
)
station_meta = pd.read_csv(STATION_META_PATH)
name_map = station_meta.set_index('station_id')['station_name'].to_dict()

def short_name(sid, maxlen=20):
    n = name_map.get(sid, str(sid))
    return (n[:maxlen] + '...') if len(n) > maxlen else n

print(f"Loaded: {len(df):,} rows, {df['station_id'].nunique()} stations")

Loaded: 3,227,633 rows, 328 stations


## Station x month day/night aggregation table

In [3]:
g = df.groupby(['station_id', 'month']).agg(
    total_day_count=('day_count_weighted', 'sum'),
    total_night_count=('night_count_weighted', 'sum'),
    day_hours_weighted=('daylight_fraction', 'sum'),
).reset_index()

night_hours = df.groupby(['station_id', 'month'])['daylight_fraction'].apply(lambda v: (1 - v).sum())
g = g.set_index(['station_id', 'month'])
g['night_hours_weighted'] = night_hours
g = g.reset_index()

g['avg_day_count_per_hour'] = g['total_day_count'] / g['day_hours_weighted']
g['avg_night_count_per_hour'] = g['total_night_count'] / g['night_hours_weighted']

# ratio: mark night_count=0 (division by zero) cases as NaN, not infinity
with np.errstate(divide='ignore', invalid='ignore'):
    ratio = g['avg_day_count_per_hour'] / g['avg_night_count_per_hour']
ratio[np.isinf(ratio)] = np.nan
g['day_night_ratio'] = ratio

# Combinations with too few samples (<10) get their ratio marked unreliable -> NaN (see note #3 above)
low_sample = (g['total_day_count'] < LOW_SAMPLE_THRESHOLD) | (g['total_night_count'] < LOW_SAMPLE_THRESHOLD)
g['day_night_ratio_reliable'] = ~low_sample
g.loc[low_sample, 'day_night_ratio'] = np.nan

g = g.merge(station_meta[['station_id', 'station_name', 'longitude_wgs84', 'latitude_wgs84']],
            on='station_id', how='left')
g = g[['station_id', 'station_name', 'longitude_wgs84', 'latitude_wgs84', 'month',
       'total_day_count', 'total_night_count', 'day_hours_weighted', 'night_hours_weighted',
       'avg_day_count_per_hour', 'avg_night_count_per_hour', 'day_night_ratio', 'day_night_ratio_reliable']]

g.to_csv(STATION_MONTH_OUT, index=False)
print(f"Written: {STATION_MONTH_OUT} ({len(g):,} rows, theoretical max 328x14={328*14})")
print(f"Combinations flagged unreliable due to low sample size: {low_sample.sum()}")
g.head()

Written: output/station_month_day_night.csv (4,461 rows, theoretical max 328x14=4592)
Combinations flagged unreliable due to low sample size: 133


,station_id,station_name,longitude_wgs84,latitude_wgs84,month,total_day_count,total_night_count,day_hours_weighted,night_hours_weighted,avg_day_count_per_hour,avg_night_count_per_hour,day_night_ratio,day_night_ratio_reliable
0,5677,Zählfeld J_36.1_1_I (veraltet),9.999514,53.580505,2025-01,11874.244433,13593.755567,252.252771,491.747229,47.072801,27.643787,1.702835,True
1,5677,Zählfeld J_36.1_1_I (veraltet),9.999514,53.580505,2025-02,17285.202328,9761.797672,275.445507,396.554493,62.753619,24.616535,2.549247,True
2,5677,Zählfeld J_36.1_1_I (veraltet),9.999514,53.580505,2025-03,36477.750782,6205.249218,368.817411,375.182589,98.904633,16.539278,5.979985,True
3,5677,Zählfeld J_36.1_1_I (veraltet),9.999514,53.580505,2025-04,45443.223666,12261.776334,421.249268,298.750732,107.877276,41.043502,2.628364,True
4,5677,Zählfeld J_36.1_1_I (veraltet),9.999514,53.580505,2025-05,53850.636179,4318.363821,493.920737,250.079263,109.026879,17.267980,6.313818,True


## Network-wide daily Day vs Night line chart

In [4]:
daily = df.groupby('date').agg(
    total_day_count=('day_count_weighted', 'sum'),
    total_night_count=('night_count_weighted', 'sum'),
    day_hours_weighted=('daylight_fraction', 'sum'),
).reset_index()
daily_night_hours = df.groupby('date')['daylight_fraction'].apply(lambda v: (1 - v).sum())
daily = daily.set_index('date')
daily['night_hours_weighted'] = daily_night_hours
daily = daily.reset_index()
daily['avg_day_count_per_hour'] = daily['total_day_count'] / daily['day_hours_weighted']
daily['avg_night_count_per_hour'] = daily['total_night_count'] / daily['night_hours_weighted']
daily['date'] = pd.to_datetime(daily['date'])
daily = daily.sort_values('date')

fig, ax = plt.subplots(figsize=(16, 6))
ax.plot(daily['date'], daily['avg_day_count_per_hour'], label='Day (avg_day_count_per_hour)', color='#e67e22', linewidth=1)
ax.plot(daily['date'], daily['avg_night_count_per_hour'], label='Night (avg_night_count_per_hour)', color='#2c3e50', linewidth=1)
ax.set_xlabel('Date'); ax.set_ylabel('Avg count per hour')
ax.set_title('Daily Day vs Night Avg Count/Hour (network-wide, all 328 stations combined, 424 days)')
ax.legend(); ax.grid(alpha=0.3)
plt.savefig(f"{CHART_DIR}/daily_day_night_line.png")
plt.show()

## Station x Month Day/Night Ratio Heatmap (log scale, diverging)

In [5]:
months = sorted(g['month'].unique())
# Station order: by average daytime volume over the whole period, descending (busiest at the top)
station_order = g.groupby('station_id')['avg_day_count_per_hour'].mean().sort_values(ascending=False).index.tolist()

pivot_ratio = g.pivot(index='station_id', columns='month', values='day_night_ratio').reindex(station_order)
log_data = np.log10(pivot_ratio.values.astype(float))
masked = np.ma.masked_invalid(log_data)

cmap = plt.cm.RdBu_r.copy()
cmap.set_bad('lightgray')
vmax = np.nanmax(np.abs(log_data[np.isfinite(log_data)]))
norm = mcolors.TwoSlopeNorm(vmin=-vmax, vcenter=0, vmax=vmax)

fig, ax = plt.subplots(figsize=(9, 40))
im = ax.imshow(masked, aspect='auto', cmap=cmap, norm=norm)
ax.set_xticks(range(len(months))); ax.set_xticklabels(months, rotation=45, ha='right')
ax.set_yticks(range(len(station_order))); ax.set_yticklabels([short_name(s) for s in station_order], fontsize=6)
ax.set_title('Station x Month Day/Night Ratio (log scale, gray = insufficient sample size)', fontsize=12)
cbar = plt.colorbar(im, ax=ax, shrink=0.3)
tick_locs = [-vmax, -vmax/2, 0, vmax/2, vmax]
cbar.set_ticks(tick_locs)
cbar.set_ticklabels([f'{10**t:.2f}' for t in tick_locs])
cbar.set_label('day_night_ratio')
plt.savefig(f"{CHART_DIR}/station_month_ratio_heatmap.png")
plt.show()
print("Note: this figure is tall (328 rows) -- recommended to download and zoom in with an image viewer")

Note: this figure is tall (328 rows) -- recommended to download and zoom in with an image viewer


## Station x Month Day/Night Volume Heatmaps (log scale)

In [6]:
volume_configs = [
    ('avg_day_count_per_hour', 'YlOrRd', 'Station x Month Avg DAY Count/Hour (log scale)',
     'station_month_day_volume_heatmap.png'),
    ('avg_night_count_per_hour', 'PuBu', 'Station x Month Avg NIGHT Count/Hour (log scale)',
     'station_month_night_volume_heatmap.png'),
]

for col, cmap_name, title, fname in volume_configs:
    pivot = g.pivot(index='station_id', columns='month', values=col).reindex(station_order)
    data = pivot.values.astype(float)
    data_pos = np.where(data <= 0, np.nan, data)  # log can't handle 0, mark as missing
    masked = np.ma.masked_invalid(data_pos)

    cmap = plt.get_cmap(cmap_name).copy()
    cmap.set_bad('lightgray')
    norm = mcolors.LogNorm(vmin=np.nanmin(data_pos), vmax=np.nanmax(data_pos))

    fig, ax = plt.subplots(figsize=(9, 40))
    im = ax.imshow(masked, aspect='auto', cmap=cmap, norm=norm)
    ax.set_xticks(range(len(months))); ax.set_xticklabels(months, rotation=45, ha='right')
    ax.set_yticks(range(len(station_order))); ax.set_yticklabels([short_name(s) for s in station_order], fontsize=6)
    ax.set_title(title, fontsize=12)
    cbar = plt.colorbar(im, ax=ax, shrink=0.3)
    cbar.set_label(col + ' (log scale)')
    plt.savefig(f"{CHART_DIR}/{fname}")
    plt.show()
    print(f"{fname} done")

station_month_day_volume_heatmap.png done


station_month_night_volume_heatmap.png done


## Completion check

In [7]:
png_files = sorted([f for f in os.listdir(CHART_DIR) if f.endswith('.png')])
print(f"{len(png_files)} PNGs created (expected 4):")
for f in png_files:
    print(" -", f)
print(f"\nstation_month_day_night.csv: {len(g):,} rows")

4 PNGs created (expected 4):
 - daily_day_night_line.png
 - station_month_day_volume_heatmap.png
 - station_month_night_volume_heatmap.png
 - station_month_ratio_heatmap.png

station_month_day_night.csv: 4,461 rows
